# Warehouse Inventory Chatbot — HuggingFace API

**Stack (100% free):** LangChain · HuggingFace Inference API · FAISS · HuggingFace Embeddings

### Setup
```bash
pip install -r requirements.txt
```
Get your free token → https://huggingface.co/settings/tokens  
Paste it in `.env` as `HUGGINGFACEHUB_API_TOKEN=hf_...`

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFaceEndpoint
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document

load_dotenv()

CSV_PATH   = 'data/inventory.csv'   # ← your CSV here
LLM_MODEL  = 'mistralai/Mistral-7B-Instruct-v0.3'
# Alternatives: 'HuggingFaceH4/zephyr-7b-beta'  |  'google/flan-t5-large'
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

print('Libraries loaded.')

In [ ]:
# ── Load & enrich CSV ────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df['total_value']  = df['quantity'] * df['unit_price']
df['stock_status'] = df.apply(
    lambda r: 'LOW STOCK' if r['quantity'] <= r['reorder_level'] else 'ADEQUATE',
    axis=1
)
print(f'{len(df)} items loaded.')
df.head()

In [ ]:
# ── Convert rows to Documents ────────────────────────────────────
def df_to_documents(df):
    docs = []
    for _, row in df.iterrows():
        content = (
            f"Item ID: {row['item_id']}. Name: {row['item_name']}. "
            f"Category: {row['category']}. Quantity: {row['quantity']} {row['unit']}. "
            f"Reorder level: {row['reorder_level']}. Stock status: {row['stock_status']}. "
            f"Unit price: ${row['unit_price']:.2f}. Total value: ${row['total_value']:.2f}. "
            f"Supplier: {row['supplier']}. Location: {row['location']}. "
            f"Last updated: {row['last_updated']}."
        )
        docs.append(Document(page_content=content,
                             metadata={'item_id': row['item_id'], 'category': row['category']}))

    for cat, grp in df.groupby('category'):
        docs.append(Document(
            page_content=(
                f"Category summary — {cat}: {len(grp)} items, "
                f"total qty {grp['quantity'].sum()}, "
                f"total value ${grp['total_value'].sum():.2f}, "
                f"low-stock: {(grp['stock_status']=='LOW STOCK').sum()}."
            ),
            metadata={'item_id': 'SUMMARY', 'category': cat}
        ))

    docs.append(Document(
        page_content=(
            f"Warehouse overall: {len(df)} items, "
            f"total value ${df['total_value'].sum():.2f}, "
            f"{(df['stock_status']=='LOW STOCK').sum()} items below reorder level, "
            f"categories: {', '.join(df['category'].unique())}."
        ),
        metadata={'item_id': 'GLOBAL'}
    ))
    return docs

docs = df_to_documents(df)
print(f'{len(docs)} documents created.')

In [ ]:
# ── FAISS vector store (embeddings run locally) ──────────────────
print('Loading embedding model (first run ~90MB)...')
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)
splitter    = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits      = splitter.split_documents(docs)
vectorstore = FAISS.from_documents(splits, embeddings)
print(f'Done — {len(splits)} chunks indexed.')

In [ ]:
# ── HuggingFace Inference API LLM ────────────────────────────────
PROMPT_TEMPLATE = """<s>[INST]
You are a warehouse inventory assistant.
Answer ONLY using the context provided. Be concise and use exact numbers.
If the answer is not in the context, say "I don't have that information."
For low-stock items always mention current quantity vs reorder level.

Context:
{context}

Question: {question}
[/INST]
Answer:"""

llm = HuggingFaceEndpoint(
    repo_id=LLM_MODEL,
    huggingfacehub_api_token=os.getenv('HUGGINGFACEHUB_API_TOKEN'),
    temperature=0.1,
    max_new_tokens=512,
)

prompt = PromptTemplate(template=PROMPT_TEMPLATE, input_variables=['context', 'question'])

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=vectorstore.as_retriever(search_kwargs={'k': 6}),
    chain_type_kwargs={'prompt': prompt},
    return_source_documents=True,
)
print('Chain ready!')

In [ ]:
# ── Ask a single question ────────────────────────────────────────
question = 'Which items are low in stock?'   # ← change me

result = chain.invoke({'query': question})
print('Q:', question)
print()
print('A:', result['result'])
print()
print('--- Retrieved chunks ---')
for i, doc in enumerate(result['source_documents'], 1):
    print(f'[{i}] {doc.page_content[:120]}...')

In [ ]:
# ── Interactive chat loop ────────────────────────────────────────
print('Chat started. Type quit to stop.\n')
while True:
    user_input = input('You: ').strip()
    if not user_input:
        continue
    if user_input.lower() in ('quit', 'exit', 'q'):
        print('Goodbye!')
        break
    result = chain.invoke({'query': user_input})
    print(f"Bot: {result['result']}\n")